# Build Dimension Date
1. Find the date range
2. Generate date sequence
3. Add date attributes
4. Reorder columns
5. write the data to the gold dim_date table

In [0]:
#Imports
from pyspark.sql.functions import col,min,max,cast,explode,sequence,lit,to_date,when
from pyspark.sql.functions import year,quarter,month,dayofmonth,weekofyear,date_format


### Step1 - Find the date range



In [0]:
#i am considering these values as start_date and end_date with some buffer
start_date = "2000-01-01"
end_date = "2050-12-31"


### Step2 - Generate date sequence

In [0]:
date_df = (
    spark.range(1).select(explode(sequence(to_date(lit(start_date)),to_date(lit(end_date)))).alias("full_date"))
)

### Step3 - Add date attributes

In [0]:
dim_date_df = (
    date_df.withColumns({
        'date_key':date_format("full_date","yyyyMMdd").cast('int'),
        'year':year("full_date"),
        'quarter':quarter("full_date"),
        'month':month("full_date"),
        'week':weekofyear("full_date"),
        'day':dayofmonth("full_date"),
        'day_name':date_format("full_date","EEEE"),
        'month_name':date_format("full_date","MMMM"),
        'is_weekend':when(col("day_name").isin("Saturday","Sunday"),"Y").otherwise("N")
    })
)

### Step4 - Reorder columns

In [0]:
dim_date_final_df = dim_date_df.select(
    "date_key",
    "full_date",
    "day",
    "day_name",
    "week",
    "month",
    "month_name",
    "quarter",
    "year",
    "is_weekend"
)

### Step5 - write the data to the gold dim_date table

In [0]:
(
    dim_date_final_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","True")
        .saveAsTable("neo_bank.gold.dim_date")
)

In [0]:
%sql
select * from neo_bank.gold.dim_date